In [65]:
import sys
sys.path.insert(0, '/home/jazz/Projects/Statistical-Learning-e20452')

import pandas as pd
import numpy as np
from itables import init_notebook_mode
from open_dataset_store import quick_start
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


store = quick_start('./ExperimentResults', backend='local')

init_notebook_mode(all_interactive=True)

# import itables.options as opt
# opt.lengthMenu = [10, 25, 50]
# opt.scrollX = True

CSV_PATH = './results/state_log.csv'

df = pd.read_csv(CSV_PATH)
df = df.copy()
base_year = 2014
def _ep_to_datetime(row):
    day, hour, minute = int(row['DayOfYear']), int(row['Hour']), int(row['Minute'])
    if hour >= 24:
        day += 1
        hour -= 24
    return pd.Timestamp(year=base_year, month=1, day=1) + pd.Timedelta(days=day-1, hours=hour, minutes=minute)

df['Datetime'] = df.apply(_ep_to_datetime, axis=1)
df['timestamp'] = df['Datetime'].astype('int64') // 10**9
ZONES = ['SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']
print(f'Loaded {len(df)} timesteps, columns: {len(df.columns)}')

# list(df.columns)
# df.head(n=20)
# sum = store.get_df_summary(df, detailed=True)

Store initialised at: ./ExperimentResults (Backend: local)


Loaded 2880 timesteps, columns: 307


In [66]:

# ---------------------------------------------------------
# 1. UPDATED PERFORMANCE TABLE FUNCTION
# ---------------------------------------------------------

def get_detailed_mpc_performance_table(df, zones, hour_range=[0, 24]):
    # Filter for the specific hour range if provided
    if hour_range is not None:
        if 'Hour' in df.columns:
            start_hour, end_hour = hour_range
            mask = (df['Hour'] >= start_hour) & (df['Hour'] <= end_hour)
            df = df[mask].copy()
        elif 'Datetime' in df.columns:
            # Fallback: extract hour directly from Datetime
            start_hour, end_hour = hour_range
            hours = pd.to_datetime(df['Datetime']).dt.hour
            mask = (hours >= start_hour) & (hours <= end_hour)
            df = df[mask].copy()
        else:
            print("Warning: 'Hour' or 'Datetime' column not found in DataFrame. Using whole day.")
            
    # Comfort bounds
    bounds = {
        'Temp': (20.0, 24.0), 
        'RH': (30.0, 60.0),
        'W': (0.0, 0.012), 
        'CO2': (0.0, 1000.0)
    }
    
    def get_error_metrics(series, low, high):
        """Returns % time out of bounds, Max Deviation, and Mean Absolute Deviation (MAD)."""
        series = series.dropna() # Ignore NaNs which can skew the length
        if len(series) == 0:
            return 0, 0, 0
            
        devs = np.maximum(0, np.maximum(low - series, series - high))
        oob_mask = devs > 0
        
        pct_oob = (oob_mask.sum() / len(series)) * 100
        max_dev = devs.max()
        mad = devs.mean()
        
        return pct_oob, max_dev, mad

    performance_data = []
    
    for zone in zones:
        status_col = f"{zone}_MPC_Status"
        solve_time_col = f"{zone}_MPC_Time_ms"
        
        if status_col not in df.columns: 
            continue
            
        total = len(df)
        if total == 0:
            continue
            
        success = len(df[df[status_col].isin([1, 2])])
        infeasible = len(df[df[status_col].isin([3, 4])])
        unbounded = len(df[df[status_col].isin([5, 6])])
        max_iter = len(df[df[status_col] == 7])
        
        T_series = df.get(f"{zone}_Temp_C", pd.Series(dtype=float))
        RH_series = df.get(f"{zone}_RH_pct", pd.Series(dtype=float))
        W_series = df.get(f"{zone}_W_kg_kg", pd.Series(dtype=float))  
        C_series = df.get(f"{zone}_CO2_ppm", pd.Series(dtype=float))
        
        t_pct, t_max, t_mad = get_error_metrics(T_series, *bounds['Temp'])
        rh_pct, rh_max, rh_mad = get_error_metrics(RH_series, *bounds['RH'])
        w_pct, w_max, w_mad = get_error_metrics(W_series, *bounds['W'])
        c_pct, c_max, c_mad = get_error_metrics(C_series, *bounds['CO2'])
        
        mean_solve = df[solve_time_col].mean() if solve_time_col in df.columns else 0
        max_solve = df[solve_time_col].max() if solve_time_col in df.columns else 0
        
        performance_data.append({
            "Zone": zone,
            "Success (%)": round((success / total) * 100, 1),
            "Infeas (%)": round((infeasible / total) * 100, 1),
            "Unbnd (%)": round((unbounded / total) * 100, 1),
            "MaxIter (%)": round((max_iter / total) * 100, 1),
            "Mean Solve (ms)": round(mean_solve, 2),
            "Max Solve (ms)": round(max_solve, 2),
            
            "T OOB (%)": round(t_pct, 1),
            "T MaxDev": round(t_max, 2),
            "T MAD": round(t_mad, 3),
            
            "RH OOB (%)": round(rh_pct, 1),
            "RH MaxDev": round(rh_max, 1),
            "RH MAD": round(rh_mad, 2),

            "W OOB (%)": round(w_pct, 1),
            "W MaxDev": round(w_max, 4),
            "W MAD": round(w_mad, 5),
            
            "CO2 OOB (%)": round(c_pct, 1),
            "CO2 MaxDev": round(c_max, 0),
            "CO2 MAD": round(c_mad, 1)
        })
        
    return pd.DataFrame(performance_data)


# Global Styles
SOURCE_STYLES = {
    "Zone":     {"color": "#1f77b4", "dash": "solid",   "width": 2},    
    "Outside":  {"color": "#ff7f0e", "dash": "dash",    "width": 0.5},  
    "Supply":   {"color": "#2ca02c", "dash": "dash",    "width": 0.5},  
    "Other_1":  {"color": "#FFBE91", "dash": "solid",   "width": 2},    
    "Setpoint": {"color": "#CFEBFF", "dash": "dot",     "width": 0.5},  
    "EKF":      {"color": "#FF00FF", "dash": "dot",    "width": 0.5}, 
}

def build_zone_subplots(df, zone_name, subplot_config, source_styles=SOURCE_STYLES, row_height=250):
    total_rows = len(subplot_config)
    
    fig = make_subplots(
        rows=total_rows,
        cols=1,
        shared_xaxes=False,
        subplot_titles=[panel["title"] for panel in subplot_config],
        specs=[[{"secondary_y": True}]] * total_rows
    )
    
    for row_idx, panel in enumerate(subplot_config, start=1):
        
        # --- 1. Add Data Traces ---
        for trace_info in panel["traces"]:
            col_name = trace_info["col"]
            source_type = trace_info.get("source", "Zone")
            trace_type = trace_info.get("type", "line")
            
            if col_name in df.columns:
                display_name = trace_info.get("name", col_name)
                is_secondary = trace_info.get("secondary_y", False)
                
                if trace_type == "status":
                    fig.add_trace(
                        go.Scatter(
                            x=df["Datetime"], y=df[col_name],
                            name=display_name, mode="markers",
                            marker=dict(
                                color=df[col_name],
                                colorscale=trace_info.get("colorscale", [[0, "green"], [0.5, "yellow"], [1, "red"]]),
                                cmin=trace_info.get("cmin", 1), cmax=trace_info.get("cmax", 7),
                                size=6, symbol="square"
                            ),
                            legendgroup=str(row_idx),
                            legendgrouptitle_text=f"<b>{panel['title']}</b>"
                        ),
                        row=row_idx, col=1, secondary_y=is_secondary
                    )
                else:
                    base_style = source_styles.get(source_type, {"color": "black", "dash": "solid", "width": 1})
                    line_color = trace_info.get("color", base_style["color"])
                    line_dash = trace_info.get("dash", base_style["dash"])
                    line_width = trace_info.get("width", base_style["width"])
                    
                    fig.add_trace(
                        go.Scatter(
                            x=df["Datetime"], y=df[col_name], name=display_name,
                            legendgroup=str(row_idx), legendgrouptitle_text=f"<b>{panel['title']}</b>",
                            line=dict(color=line_color, dash=line_dash, width=line_width)
                        ),
                        row=row_idx, col=1, secondary_y=is_secondary
                    )

        # --- 2. Draw Expected Range Horizontal Band ---
        if "expected_range" in panel:
            ymin, ymax = panel["expected_range"]
            range_label = panel.get("expected_label", "Expected Range")
            range_color = panel.get("range_color", "rgba(46, 204, 113, 0.15)") # Transparent green
            
            fig.add_hrect(
                y0=ymin, y1=ymax, fillcolor=range_color,
                line_width=0, layer="below", row=row_idx, col=1, secondary_y=False
            )
            
            # --- 3. NEW: Draw Transparent Red Vertical Background for Out of Bounds ---
            if "eval_col" in panel and panel["eval_col"] in df.columns:
                col_to_check = panel["eval_col"]
                # Mask where values are OUT of bounds
                oob = (df[col_to_check] < ymin) | (df[col_to_check] > ymax)
                is_oob = oob.values
                
                if is_oob.any():
                    # Find continuous blocks where it's out of range to draw rectangles
                    transitions = np.where(is_oob[:-1] != is_oob[1:])[0]
                    start_idx = 0
                    intervals = []
                    
                    for t in transitions:
                        if is_oob[start_idx]: 
                            intervals.append((df["Datetime"].iloc[start_idx], df["Datetime"].iloc[t + 1]))
                        start_idx = t + 1
                        
                    if is_oob[start_idx]:
                        intervals.append((df["Datetime"].iloc[start_idx], df["Datetime"].iloc[-1]))
                        
                    # Render the red transparent blocks
                    for start_t, end_t in intervals:
                        fig.add_vrect(
                            x0=start_t, x1=end_t,
                            fillcolor="rgba(255, 0, 0, 0.12)", # Faint red background
                            layer="below", line_width=0, row=row_idx, col=1, secondary_y=False
                        )
                
        # --- 4. Set Y-axis Titles and Ranges ---
        primary_y_kwargs = {"title_text": panel.get("y_label", "")}
        if "y_range" in panel:
            primary_y_kwargs["range"] = panel["y_range"]
        fig.update_yaxes(**primary_y_kwargs, row=row_idx, col=1, secondary_y=False)
        
        if "secondary_y_label" in panel or "secondary_y_range" in panel:
            secondary_y_kwargs = {}
            if "secondary_y_label" in panel:
                secondary_y_kwargs["title_text"] = panel["secondary_y_label"]
            if "secondary_y_range" in panel:
                secondary_y_kwargs["range"] = panel["secondary_y_range"]
            fig.update_yaxes(**secondary_y_kwargs, row=row_idx, col=1, secondary_y=True)
            
    fig.update_layout(
        height=max(400, row_height * total_rows),
        title_text=f"{zone_name} Dashboard", hovermode="x unified",
        showlegend=True, legend=dict(groupclick="toggleitem", tracegroupgap=15), margin=dict(r=150)
    )
    
    return fig


# ZONE MONITOR


* **1 (`SOLVED`)**: The MPC successfully converged and found an optimal set of control inputs.
* **2 (`SOLVED_INACCURATE`)**: The solver found a viable solution but stopped due to numerical tolerances before reaching maximum accuracy.
* **3 (`PRIMAL_INFEASIBLE`)**: The solver mathematically proved that your constraints cannot be satisfied simultaneously (e.g., conflicting temperature bounds).
* **4 (`PRIMAL_INFEASIBLE_INACCURATE`)**: The problem is likely impossible to solve, but the solver stopped due to numerical inaccuracies before proving it completely.
* **5 (`DUAL_INFEASIBLE`)**: The problem is unbounded, meaning the objective function can scale infinitely (this usually indicates a missing constraint in the matrix setup).
* **6 (`DUAL_INFEASIBLE_INACCURATE`)**: The problem is likely unbounded, but the solver was forced to stop early due to numerical issues.
* **7 (`MAX_ITER`)**: The solver hit its iteration limit (which you have configured to 50,000) without successfully converging to a solution.


In [67]:
ZONES = ["SPACE1-1"]
# ZONES = ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]

In [68]:
# DAY TIME
perf_df_daytime = get_detailed_mpc_performance_table(df, ZONES, hour_range=[8, 18])
perf_df_daytime

Loading ITables v2.8.1 from the init_notebook_mode cell... (need help?)


In [69]:
# FULL DAY
perf_df= get_detailed_mpc_performance_table(df, ZONES )
perf_df

Loading ITables v2.8.1 from the init_notebook_mode cell... (need help?)


In [70]:
# ZONE PLOT
for i in ZONES:
    zone = i

    config = [
        {
            "title": "MPC Solver Status",
            "y_label": "OSQP Status Code",
            "y_range": [0, 8],
            "traces": [
                {
                    "col": f"{zone}_MPC_Status", 
                    "name": "Status (1=Green, 7=Red)",
                    "type": "status",
                    "colorscale": [[0.0, "green"], [0.5, "yellow"], [1.0, "red"]],
                    "cmin": 1,
                    "cmax": 7
                }
            ]
        },
        {
            "title": f"Temperature",
            "y_label": "Temperature (°C)",
            "y_range": [5, 35],
            "expected_range": [20, 24], 
            "eval_col": f"{zone}_Temp_C",   # <--- Added this to trigger red highlights
            "range_color": "rgba(52, 152, 219, 0.05)",
            "expected_label": "Comfort Zone",
            "traces": [
                {"col": f"{zone}_Temp_C", "source": "Zone",    "name": "Zone Temp"},
                {"col": "Out_Temp_C",     "source": "Outside", "name": "Outdoor Temp"},
                {"col": "Fan_Out_Temp_C", "source": "Supply",  "name": "Supply Temp"},
                {"col": f"{zone}_EKF_x_T_in",   "source": "EKF",     "name": "Estimated T_in"},
            ]
        },
        {
            "title": f"Humidity Ratio",
            "y_label": "Humidity (kg/kg)",
            "expected_range": [0, 0.012],
            "eval_col": f"{zone}_W_kg_kg",   # <--- ADDED THIS LINE to trigger red highlight
            "range_color": "rgba(52, 152, 219, 0.05)",
            "expected_label": "Target W Band",
            "traces": [
                {"col": f"{zone}_W_kg_kg", "source": "Zone",    "name": "Zone W"},
                {"col": "Out_W_kg_kg",     "source": "Outside", "name": "Outdoor W"},
                {"col": "Fan_Out_W_kg_kg", "source": "Supply",  "name": "Supply W"},
                {"col": f"{zone}_EKF_x_W_in",   "source": "EKF",     "name": "Estimated Humidity"},
            ]
        },
        {
            "title": f"Relative Humidity",
            "y_label": "Relative Humidity (%)",
            "y_range": [0, 100],
            "expected_range": [30, 60],
            "eval_col": f"{zone}_RH_pct",   # <--- Added this to trigger red highlights
            "range_color": "rgba(52, 152, 219, 0.05)",
            "expected_label": "Target RH Band",
            "traces": [
                {"col": f"{zone}_RH_pct", "source": "Zone",    "name": "Zone RH"},
                {"col": "Out_RH_pct",     "source": "Outside", "name": "Outdoor RH"},
                {"col": "Fan_Out_RH_pct", "source": "Supply",  "name": "Supply RH"},
            ]
        },
        {
            "title": f"CO2 & Occupancy",
            "y_label": "CO2 (ppm)",
            "secondary_y_label": "Occupants",  
            "y_range": [0, 1500],
            "expected_range": [0, 1000],
            "eval_col": f"{zone}_CO2_ppm",  # <--- Added this to trigger red highlights
            "expected_label": "Acceptable CO2",
            "range_color": "rgba(52, 152, 219, 0.05)",
            "traces": [
                {"col": f"{zone}_CO2_ppm", "source": "Zone",    "name": "Zone CO2"},
                {"col": "Out_CO2_ppm",     "source": "Outside", "name": "Outdoor CO2"},
                {"col": "Fan_Out_CO2_ppm", "source": "Supply",  "name": "Supply CO2"},
                {"col": f"{zone}_Occupants", "source": "Setpoint",  "name": "No of Occupancy", "secondary_y": True}, 
            ]
        },
        {
            "title": f"Occupancy & Humidity Disturbance (d_W)",
            "y_label": "Occupants",  
            "secondary_y_label": "Hum Disturbance (d_W)",
            "traces": [
                {"col": f"{zone}_Occupants", "source": "Zone",  "name": "No of Occupancy"}, 
                {"col": f"{zone}_EKF_x_N_occ",  "source": "EKF",     "name": "Estimated Occupancy"},
                {"col": f"{zone}_EKF_x_d_W",    "source": "Setpoint",  "name": "Estimated d_W", "secondary_y": True},
            ]
        },
        {
            "title": "Equipment Load & Thermal Disturbance (d_T)",
            "y_label": "Temp Disturbance (d_T)",
            "secondary_y_label": "Equipment Load (W)",
            "traces": [
                {"col": f"{zone}_EquipLoad_W",  "source": "Other_1",  "name": "True Equip Load (W)", "secondary_y": True},
                {"col": f"{zone}_EKF_x_d_T",    "source": "EKF",     "name": "Estimated d_T"},
            ]
        },
        {
            "title": f"VAV Flow & Equipment Status",
            "y_label": "Mass Flow (kg/s)", 
            "secondary_y_label": "Power (W) / Temp (°C)",
            "traces": [
                {"col": f"{zone}_VAV_Flow_kg_s", "source": "Zone",     "name": "VAV Flow"},
                {"col": f"{zone}_Flow_SP_kg_s",  "source": "Setpoint", "name": "Flow Setpoint"},
            ]
        },
    ]

    fig = build_zone_subplots(df, zone, config)
    fig.show()


# AHU MONITOR